# AN-RA iterate500 T4 Training
Canonical bootstrap, preflight, 500M-class frontier training, resume, and ThirdEye evaluation. The trainer restores from MyDrive checkpoints first, then visible shared Drive locations / Shared-with-me when available. Secrets must be supplied through Colab secrets/environment variables, never notebook cells.

In [ ]:
from google.colab import drive
from pathlib import Path

def mount_drive_or_stop():
    for force in (False, True):
        try:
            drive.mount('/content/drive', force_remount=force)
            if Path('/content/drive/MyDrive').exists():
                print('[Drive] mounted at /content/drive')
                return
        except Exception as exc:
            print(f'[Drive] mount attempt force_remount={force} failed: {type(exc).__name__}: {exc}')
    raise RuntimeError('Google Drive auth failed before training. In Colab: Runtime -> Disconnect and delete runtime, reload the notebook, sign into the correct Google account, then run again. Training needs Drive for checkpoint resume/save.')

mount_drive_or_stop()

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Wrong Colab runtime. Choose Runtime -> Change runtime type -> T4 GPU. TPU v5e/CPU should use the TPU notebook.')
print('gpu:', torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
REPO = '/content/An-Ra-the-new-AGI'
BRANCH = 'iterate500'
![ -d "$REPO/.git" ] || git clone --branch "$BRANCH" --single-branch https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git "$REPO"
%cd $REPO
!git fetch origin "$BRANCH"
!git checkout "$BRANCH"
!git pull --ff-only origin "$BRANCH"
import os
from pathlib import Path
PIP_CACHE = Path('/content/.cache/pip')
PIP_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['PIP_CACHE_DIR'] = str(PIP_CACHE)
os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
os.environ['PIP_NO_INPUT'] = '1'
!python scripts/colab_bootstrap.py --repo "$REPO" --drive-root /content/drive/MyDrive/AnRa --install --install-thirdeye --model-size frontier

In [ ]:
%cd /content/An-Ra-the-new-AGI
import os
from pathlib import Path
from training.shared_checkpoint import restore_shared_checkpoint

# This branch continues the existing 500M experiment. Set False only for a truly new model.
REQUIRE_RESUME = True
CHECKPOINT = Path('anra_frontier_500m.pt')
os.environ['ANRA_SHARED_DRIVE_API'] = '1'
# This run has one Drive checkpoint only: the owner master in My Drive or its editable Shared with me view.
# Training stops rather than creating a private or versioned checkpoint copy.
os.environ['ANRA_REQUIRE_SHARED_MASTER'] = '1'
os.environ['ANRA_REQUIRE_RESUME'] = '1' if REQUIRE_RESUME else '0'
source = restore_shared_checkpoint(CHECKPOINT)
if CHECKPOINT.exists():
    print(f'[Resume Check] READY: {CHECKPOINT} ({CHECKPOINT.stat().st_size / 1024**3:.2f} GB), source={source}')
elif REQUIRE_RESUME:
    raise RuntimeError('Resume checkpoint was not found. Training is intentionally stopped so it cannot restart at step 1. Confirm the same Google account owns or can access anra_frontier_500m.pt, then rerun this cell.')
else:
    print('[Resume Check] No checkpoint found. Fresh training is explicitly allowed.')

In [ ]:
%cd /content/An-Ra-the-new-AGI
# Immutable native corpus profile. Use '15gb' only when Drive space is constrained.
DATA_PROFILE = '30gb'
FORCE_DATA_REBUILD = False
data_args = '--force-rebuild' if FORCE_DATA_REBUILD else ''
!python scripts/colab_prepare_data.py --repo /content/An-Ra-the-new-AGI --profile $DATA_PROFILE --drive-root /content/drive/MyDrive/AnRa $data_args
!python -m data.causal_corpus
!python scripts/show_thirdeye_summary.py --profile quick --without-model

In [ ]:
import os
# Saved into every checkpoint. A future mismatch is rejected before training.
os.environ['ANRA_DATA_PROFILE'] = DATA_PROFILE
os.environ['ANRA_ALLOW_DATA_PROFILE_CHANGE'] = '1'  # explicit native continuation campaign
os.environ['ANRA_TRAINING_DATA_LAYOUT'] = 'raw_causal_shards_v1'
os.environ.setdefault('ANRA_CHECKPOINT_EVERY_MIN', '45')
os.environ.setdefault('ANRA_THIRDEYE_INTELLIGENCE', '1')
os.environ.setdefault('ANRA_THIRDEYE_SAMPLE_EVERY', '50')
SESSION_MINUTES = 180
CONTINUATION_PHASE = 'A'
TOKEN_MANIFEST = f'output/v2/data_manifests/native_foundation_v3/{DATA_PROFILE}/manifest.json'
VALIDATION_MANIFEST = f'output/v2/data_manifests/native_foundation_v3/{DATA_PROFILE}/validation/manifest.json'
print('LOSS VIEW: this cell is pure training. Watch the step/loss/best lines here.')
!python scripts/build_brain.py --data_path training_data/anra_training.txt --checkpoint_path anra_frontier_500m.pt --model-size frontier --batch_size 1 --optimizer adafactor --max_minutes $SESSION_MINUTES --training-layout raw_causal_shards_v1 --token-shard-manifest $TOKEN_MANIFEST --validation-shard-manifest $VALIDATION_MANIFEST --continuation-phase $CONTINUATION_PHASE

In [ ]:
%cd /content/An-Ra-the-new-AGI
print('THIRD EYE VIEW: evidence dashboard after training. This is separate from loss.')
!python scripts/show_thirdeye_summary.py --profile quick --without-model

In [ ]:
%cd /content/An-Ra-the-new-AGI
import os
from pathlib import Path

CHECKPOINT = Path('anra_frontier_500m.pt')
if not CHECKPOINT.exists():
    drive_checkpoint = Path('/content/drive/MyDrive/AnRa/v2/checkpoints/anra_frontier_500m.pt')
    if drive_checkpoint.exists():
        CHECKPOINT = drive_checkpoint

if not CHECKPOINT.exists():
    raise RuntimeError('No frontier checkpoint found. Run the resume/training cells first, or confirm Drive access to anra_frontier_500m.pt.')

os.environ['ANRA_MODEL_PROFILE'] = 'frontier'
os.environ['ANRA_CHECKPOINT_PATH'] = str(CHECKPOINT.resolve())

print(f'FRONTIER CHECKPOINT PROOF: {CHECKPOINT} ({CHECKPOINT.stat().st_size / 1024**3:.2f} GB)')
!python scripts/check_frontier_checkpoint.py --checkpoint "$ANRA_CHECKPOINT_PATH" --json-out output/v2/frontier_checkpoint_proof.json

print('FRONTIER SMOKE CHAT: loading the trained checkpoint and saving trace JSONL.')
!python scripts/chat_frontier.py --checkpoint "$ANRA_CHECKPOINT_PATH" --suite smoke --output output/v2/frontier_chat_traces.jsonl

print('Artifacts written:')
print('  output/v2/frontier_checkpoint_proof.json')
print('  output/v2/frontier_chat_traces.jsonl')

In [ ]:
%cd /content/An-Ra-the-new-AGI
import os
import subprocess
import sys
import time
from pathlib import Path
from urllib.request import urlopen

CHECKPOINT = Path('anra_frontier_500m.pt')
if not CHECKPOINT.exists():
    drive_checkpoint = Path('/content/drive/MyDrive/AnRa/v2/checkpoints/anra_frontier_500m.pt')
    if drive_checkpoint.exists():
        CHECKPOINT = drive_checkpoint
if not CHECKPOINT.exists():
    raise RuntimeError('No frontier checkpoint found. Run resume/training first or confirm Drive access.')

os.environ['ANRA_MODEL_PROFILE'] = 'frontier'
os.environ['ANRA_CHECKPOINT_PATH'] = str(CHECKPOINT.resolve())
os.environ['ANRA_SERVICE_MODE'] = 'development'
os.environ.pop('ANRA_OWNER_TOKEN', None)

print(f'Launching An-Ra developer UI with checkpoint: {CHECKPOINT}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=False)
subprocess.run(['fuser', '-k', '8000/tcp'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['fuser', '-k', '5173/tcp'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

backend_env = os.environ.copy()
backend = subprocess.Popen(
    [sys.executable, 'app.py', '--host', '127.0.0.1', '--port', '5173'],
    env=backend_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

for attempt in range(180):
    try:
        with urlopen('http://127.0.0.1:5173/status', timeout=2) as response:
            print('Backend ready:', response.status)
            break
    except Exception:
        if backend.poll() is not None:
            raise RuntimeError('Backend exited before /status became ready.')
        time.sleep(1)
else:
    raise TimeoutError('Backend did not become ready on port 5173.')

from google.colab.output import eval_js
from IPython.display import HTML, IFrame, display

base_url = eval_js('google.colab.kernel.proxyPort(5173)')
ui_url = base_url.rstrip('/') + '/developer'
print('Open the dashboard:', ui_url)
display(HTML(f'<a href="{ui_url}" target="_blank">Open An-Ra Developer UI</a>'))
display(IFrame(ui_url, width='100%', height=820))
print('Use DASHBOARD to chat. Use MATRIX to inspect checkpoint, runtime, HAL, sessions, and raw payloads.')

In [ ]:
# RUN ONLY THIS CELL FOR CHAT + MATRIX. IT DOES NOT TRAIN THE MODEL.
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json
import os
import subprocess
import sys
import time
from pathlib import Path
from urllib.request import urlopen

REPO = Path('/content/An-Ra-the-new-AGI')
BRANCH = 'iterate500'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, 'scripts/colab_bootstrap.py', '--repo', str(REPO), '--drive-root', '/content/drive/MyDrive/AnRa', '--install', '--model-size', 'frontier'], check=True)

from training.shared_checkpoint import restore_shared_checkpoint
checkpoint = REPO / 'anra_frontier_500m.pt'
source = restore_shared_checkpoint(checkpoint, checkpoint.name)
if not checkpoint.exists():
    fallback = Path('/content/drive/MyDrive/AnRa/v2/checkpoints/anra_frontier_500m.pt')
    if fallback.exists():
        checkpoint = fallback
if not checkpoint.exists():
    raise RuntimeError('The trained anra_frontier_500m.pt checkpoint was not found in Drive. No training was started.')

os.environ['ANRA_MODEL_PROFILE'] = 'frontier'
os.environ['ANRA_CHECKPOINT_PATH'] = str(checkpoint.resolve())
os.environ['ANRA_SERVICE_MODE'] = 'development'
os.environ.pop('ANRA_OWNER_TOKEN', None)
print(f'Checkpoint: {checkpoint} ({checkpoint.stat().st_size / 1024**3:.2f} GB), source={source}')
subprocess.run([sys.executable, 'scripts/check_frontier_checkpoint.py', '--checkpoint', str(checkpoint), '--json-out', 'output/v2/frontier_checkpoint_proof.json'], check=True)

subprocess.run(['fuser', '-k', '5173/tcp'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
log_path = Path('/content/anra_developer_ui.log')
log_handle = log_path.open('w', encoding='utf-8')
backend = subprocess.Popen([sys.executable, 'app.py', '--host', '127.0.0.1', '--port', '5173'], env=os.environ.copy(), stdout=log_handle, stderr=subprocess.STDOUT)
status = None
for _ in range(240):
    try:
        with urlopen('http://127.0.0.1:5173/status', timeout=3) as response:
            status = json.loads(response.read().decode('utf-8'))
            break
    except Exception:
        if backend.poll() is not None:
            log_handle.flush()
            raise RuntimeError('Backend exited during startup. Log: ' + log_path.read_text(encoding='utf-8', errors='replace')[-4000:])
        time.sleep(1)
if status is None:
    raise TimeoutError(f'Backend did not become ready. Inspect {log_path}.')
print('Backend ready:', json.dumps({key: status.get(key) for key in ('service_status', 'bundle_status', 'quality_status', 'device', 'param_count')}, indent=2))

try:
    with urlopen('http://127.0.0.1:5173/diagnostics/release-evidence', timeout=300) as response:
        release_report = json.loads(response.read().decode('utf-8'))
    print('KV cache parity:', release_report.get('cache', {}).get('verified'))
    print('Session isolation:', release_report.get('session_isolation', {}).get('verified'))
    print('Release evidence:', json.dumps(release_report.get('evidence', {}), indent=2))
except Exception as exc:
    print('Release diagnostics incomplete; cache remains disabled until parity passes:', exc)

from google.colab.output import eval_js
from IPython.display import HTML, IFrame, display
base_url = eval_js('google.colab.kernel.proxyPort(5173)')
ui_url = base_url.rstrip('/') + '/developer'
display(HTML(f'<p><a href="{ui_url}" target="_blank" style="font-size:18px;font-weight:700">Open An-Ra Developer UI</a></p>'))
display(IFrame(ui_url, width='100%', height=900))
print('Dashboard = chat. Matrix = exact prompt, token allocation, checkpoint proof, subsystem telemetry, HAL/ESV state, stop reason, and evaluation gates.')
print('In MATRIX: run Rollback drill, 200-prompt gate, Integration probe, then Full promotion eval. The full evaluation runs in the backend; use Review outputs when it finishes.')